In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
import os
from os.path import join
import pandas as pd
import numpy as np
from itertools import product
from typing import List, Dict, Tuple, Any, Union
from mlex.evaluation.plotter import EvaluationPlotter 

MODELS = ['RNN', 'LSTM', 'GRU', 'BILSTM'] 
LENGTHS = ['10', '20', '30', '40', '50']
thresholds_list = ['f1max']
iterations = 10
num_layers = 1
hidden_size = 10
sequences_compositions = ['temporal', 'Feature_individual', 'Feature_account']

BASE_ROOT = '/data/isa/mlex/experiments/pcpe4/' 
IN_DIST_SUFFIX = 'results_ci_withcontext_train3_test3'
OUT_DIST_SUFFIX = 'results_ci_withcontext_train3_test4'
METRIC_TO_EXTRACT = 'auc_roc' 

DISTRIBUTION_MAPPING = {
    IN_DIST_SUFFIX: 'In-Distribution (ID)',
    OUT_DIST_SUFFIX: 'Out-of-Distribution (OOD)',
}
CONTEXT_MAPPING_EN = {
    'temporal': 'Temporal Context',
    'Feature_individual': 'Feature (Individual) Context',
    'Feature_account': 'Feature (Account) Context',
}


def generate_ids(model: str, length: str, seq_comp: str, num_layers: int = 1, hidden_size: int = 10, iterations: int = 10) -> List[str]:
    """Gera a lista de model_ids para um dado SequenceLength e Composição."""
    
    ids = [
        f"{model}_Layers-{num_layers}_HiddenSize-{hidden_size}_SequenceLength-{length}_{seq_comp}_{threshold}_Iteration-{i+1}"
        for threshold, i in product(thresholds_list, range(iterations))
    ]
    return ids

def get_metric_ci_for_table(plotter: EvaluationPlotter, model_ids: List[str], metric: str) -> Tuple[float, float]:
    """Extrai a média e o desvio padrão da métrica do DataFrame carregado."""
    
    if not model_ids or plotter.df.empty:
        return np.nan, np.nan

    df = plotter.df
    
    vals = []
    
    try:
        if not df.empty:
             vals = [
                float(ser.iloc[0])
                for mid in model_ids
                for ser in [df.loc[df['model_id'] == mid, metric]]
                if not ser.empty
            ]
    except Exception as e:
        print(f"Error extracting metric from data: {e}")
        return np.nan, np.nan
    
    if not vals:
        return np.nan, np.nan
    
    mean = np.mean(vals)
    std_auc = np.std(vals) 
    return mean, std_auc


def generate_results_dataframe(plotter_class: Any):
    results = []
    
    for folder_suffix, display_dist in DISTRIBUTION_MAPPING.items():
        for model in MODELS:
            
            path = join(BASE_ROOT, model, folder_suffix)
            parquet_path = f"{path}/evaluation.parquet"
            
            try:
                plotter = plotter_class(parquet_path)
                
              
                if plotter.df.empty:
                    print(f"Skipping model {model} in {display_dist} due to empty data.")
                    continue 

                for length in LENGTHS:
                    for seq_comp in sequences_compositions:
                        model_ids = generate_ids(model, length, seq_comp)
                        mean, std = get_metric_ci_for_table(plotter, model_ids, metric=METRIC_TO_EXTRACT)
                        
                        if not np.isnan(mean):
                            formatted_value = f"${mean:.2f} \\pm {std:.2f}$"
                            results.append({
                                'Distribution': display_dist,
                                'Model': model,
                                'Sequence Length': length,
                                'Context': CONTEXT_MAPPING_EN[seq_comp],
                                'Metric_CI': formatted_value,
                                'Raw_Mean': mean 
                            })
            except Exception as e:
                print(f"Error processing {model} in {display_dist}: {e}. Skipping this data block.")

    return pd.DataFrame(results).dropna(subset=['Raw_Mean'])


def df_to_latex_final(df: pd.DataFrame, metric_name: str = "AUC-ROC"):
    
    df = df.rename(columns={'Metric_CI': metric_name})
    
    def highlight_best(group):
        if group['Raw_Mean'].empty:
            return group
            
        idx_max = group['Raw_Mean'].idxmax()
        group.loc[idx_max, metric_name] = f"\\textbf{{{group.loc[idx_max, metric_name]}}}"
        return group

    highlighted_df = df.groupby(['Distribution', 'Model', 'Sequence Length']).apply(highlight_best).reset_index(drop=True)
    
    model_order = MODELS
    length_order = LENGTHS
    
    highlighted_df['Model'] = pd.Categorical(highlighted_df['Model'], categories=model_order, ordered=True)
    highlighted_df['Sequence Length'] = pd.Categorical(highlighted_df['Sequence Length'], categories=length_order, ordered=True)
    
    highlighted_df = highlighted_df.sort_values(['Distribution', 'Model', 'Sequence Length']).reset_index(drop=True)

    df_pivot = highlighted_df.pivot_table(
        index=['Distribution', 'Model', 'Sequence Length'],
        columns='Context',
        values=metric_name,
        aggfunc='first'
    ).reset_index()
    
    df_pivot.columns.name = None 
    
    
    rename_cols = {
        CONTEXT_MAPPING_EN['temporal']: 'Temporal', 
        
        CONTEXT_MAPPING_EN['Feature_individual']: 'Individual Context', 
        CONTEXT_MAPPING_EN['Feature_account']: 'Account Context',      
        
        'Sequence Length': 'Sequence Length'
    }
    df_pivot = df_pivot.rename(columns=rename_cols)
    
    
    df_pivot = df_pivot[['Distribution', 'Model', 'Sequence Length', 'Temporal', 'Individual Context', 'Account Context']]

    
    
    rows_per_model = len(LENGTHS) 
    rows_per_dist = len(MODELS) * rows_per_model 
    
    latex_lines = []

    
    latex_lines.append(r"\begin{table*}[!ht]")
    latex_lines.append(r"\centering")
    latex_lines.append(f"\\caption{{Classification Performance ({metric_name} $\\pm$ SD) of Sequence Models under Different Contexts and Sequence Lengths. Results are shown for In-Distribution (ID) and Out-of-Distribution (OOD) scenarios.}}")
    latex_lines.append(r"\label{tab:context_auc_full}")
   
    latex_lines.append(r"\begin{tabular}{llcccc}") 
    latex_lines.append(r"\toprule")
   
    latex_lines.append(r"Distribution & Model & Sequence Length & Temporal & Individual Context & Account Context \\")
    latex_lines.append(r"\midrule")
    
   
    current_dist = None
    current_model = None
    
    for idx, row in df_pivot.iterrows():
        line = ""
       
        if row['Distribution'] != current_dist:
            current_dist = row['Distribution']
            
            if idx > 0:
               
                latex_lines.append(r"\midrule")
                
            line += r"\multirow{" + str(rows_per_dist) + r"}{*}{" + current_dist + r"} & "
        else:
            line += r" & "
            
       
        if row['Model'] != current_model:
            current_model = row['Model']
            
           
            if idx > 0 and current_dist == df_pivot.iloc[idx-1]['Distribution'] and idx % rows_per_model == 0:
                
                latex_lines.append(r"\cline{2-6}")

            line += r"\multirow{" + str(rows_per_model) + r"}{*}{" + current_model + r"} & "
        else:
            line += r" & "
            
       
        line += f"{row['Sequence Length']} & {row['Temporal']} & {row['Individual Context']} & {row['Account Context']} \\\\"
        
        latex_lines.append(line)
    
   
    latex_lines.append(r"\bottomrule")
    latex_lines.append(r"\end{tabular}")
    latex_lines.append(r"\end{table*}")
    
    return "\n".join(latex_lines)

results_df = generate_results_dataframe(plotter_class=EvaluationPlotter)

latex_table_code = df_to_latex_final(results_df, metric_name="AUC-ROC")

print(latex_table_code)

\begin{table*}[!ht]
\centering
\caption{Classification Performance (AUC-ROC $\pm$ SD) of Sequence Models under Different Contexts and Sequence Lengths. Results are shown for In-Distribution (ID) and Out-of-Distribution (OOD) scenarios.}
\label{tab:context_auc_full}
\begin{tabular}{llcccc}
\toprule
Distribution & Model & Sequence Length & Temporal & Individual Context & Account Context \\
\midrule
\multirow{20}{*}{In-Distribution (ID)} & \multirow{5}{*}{RNN} & 10 & $0.94 \pm 0.01$ & $0.93 \pm 0.01$ & \textbf{$0.94 \pm 0.01$} \\
 &  & 20 & $0.94 \pm 0.01$ & $0.93 \pm 0.01$ & \textbf{$0.94 \pm 0.00$} \\
 &  & 30 & \textbf{$0.93 \pm 0.01$} & $0.93 \pm 0.01$ & $0.93 \pm 0.01$ \\
 &  & 40 & $0.93 \pm 0.01$ & $0.92 \pm 0.01$ & \textbf{$0.93 \pm 0.01$} \\
 &  & 50 & \textbf{$0.94 \pm 0.01$} & $0.93 \pm 0.01$ & $0.93 \pm 0.01$ \\
\cline{2-6}
 & \multirow{5}{*}{LSTM} & 10 & $0.92 \pm 0.02$ & $0.91 \pm 0.02$ & \textbf{$0.92 \pm 0.02$} \\
 &  & 20 & $0.93 \pm 0.01$ & $0.92 \pm 0.01$ & \textbf{$0.9

/tmp/ipykernel_90808/4068748431.py:138: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  highlighted_df = df.groupby(['Distribution', 'Model', 'Sequence Length']).apply(highlight_best).reset_index(drop=True)
/tmp/ipykernel_90808/4068748431.py:150: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df_pivot = highlighted_df.pivot_table(


In [ ]:
import os
from os.path import join
import pandas as pd
import numpy as np
from itertools import product
from typing import List, Tuple

from mlex.evaluation.plotter import EvaluationPlotter 

MODELS = ['RNN', 'LSTM', 'GRU', 'BILSTM']
LENGTHS = ['10', '20', '30', '40', '50']
thresholds_list = ['f1max']
iterations = 10
num_layers = 1
hidden_size = 10
sequences_compositions = ['temporal', 'Feature_individual', 'Feature_account']

BASE_ROOT = '/data/isa/mlex/experiments/pcpe4/' 

target_suffix = 'results_ci_withcontext_train3_test4' 
METRIC_TO_EXTRACT = 'auc_roc' 

CONTEXT_MAPPING = {
    'temporal': 'Temporal',
    'Feature_individual': 'Individual',
    'Feature_account': 'Account',
}


def generate_ids(model: str, length: str, seq_comp: str, iterations: int = 10) -> List[str]:
    """Gera a lista de IDs para buscar no parquet."""
    ids = [
        f"{model}_Layers-{num_layers}_HiddenSize-{hidden_size}_SequenceLength-{length}_{seq_comp}_{threshold}_Iteration-{i+1}"
        for threshold, i in product(thresholds_list, range(iterations))
    ]
    return ids

def get_metric_stats(plotter: EvaluationPlotter, model_ids: List[str], metric: str) -> Tuple[float, float]:
    """Calcula Média e Desvio Padrão para os IDs fornecidos."""
    if not model_ids or plotter.df.empty:
        return np.nan, np.nan

    df = plotter.df
    vals = []
    
    try:
        
        for mid in model_ids:
            
            row = df.loc[df['model_id'] == mid, metric]
            if not row.empty:
                vals.append(float(row.iloc[0]))
                
    except Exception as e:
        print(f"Erro ao extrair métrica: {e}")
        return np.nan, np.nan
    
    if not vals:
        return np.nan, np.nan
    
   
    return np.mean(vals), np.std(vals)


def main():
    results = []
    print(f"Iniciando processamento OOD ({target_suffix})...\n")

    for model in MODELS:
        path = join(BASE_ROOT, model, target_suffix)
        parquet_path = join(path, "evaluation.parquet")
        
        print(f"Lendo: {parquet_path}")
        
        try:
            if not os.path.exists(parquet_path):
                print(f" Arquivo não encontrado para {model}. Pulando.")
                continue

            plotter = EvaluationPlotter(parquet_path)
            
            if plotter.df.empty:
                print(f" DataFrame vazio para {model}. Pulando.")
                continue

            for length in LENGTHS:
                for seq_comp in sequences_compositions:
                    
                    model_ids = generate_ids(model, length, seq_comp, iterations=iterations)
                    
                    mean_val, sd_val = get_metric_stats(plotter, model_ids, METRIC_TO_EXTRACT)
                    
                    if not np.isnan(mean_val):
                        results.append({
                            'Model': model,
                            'Sequence_Length': length,
                            'Context': CONTEXT_MAPPING.get(seq_comp, seq_comp),
                            'Metric': METRIC_TO_EXTRACT,
                            'Mean_AUC': mean_val,
                            'SD_AUC': sd_val
                        })

        except Exception as e:
            print(f" Erro crítico ao processar modelo {model}: {e}")

    df_ood = pd.DataFrame(results)

    print("\n--- Prévia dos Resultados (OOD) ---")
    print(df_ood.head())
    print(f"Total de linhas extraídas: {len(df_ood)}")

    output_filename = 'ood_aucroc_results.csv'
    df_ood.to_csv(output_filename, index=False)
    print(f"\nArquivo salvo com sucesso: {output_filename}")

if __name__ == "__main__":
    main()

/data/isa/mlex/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Iniciando processamento OOD (results_ci_withcontext_train3_test4)...

Lendo: /data/isa/mlex/experiments/pcpe4/RNN/results_ci_withcontext_train3_test4/evaluation.parquet
Lendo: /data/isa/mlex/experiments/pcpe4/LSTM/results_ci_withcontext_train3_test4/evaluation.parquet
Lendo: /data/isa/mlex/experiments/pcpe4/GRU/results_ci_withcontext_train3_test4/evaluation.parquet
Lendo: /data/isa/mlex/experiments/pcpe4/BILSTM/results_ci_withcontext_train3_test4/evaluation.parquet

--- Prévia dos Resultados (OOD) ---
  Model Sequence_Length     Context   Metric  Mean_AUC    SD_AUC
0   RNN              10    Temporal  auc_roc  0.647793  0.036431
1   RNN              10  Individual  auc_roc  0.670055  0.040124
2   RNN              10     Account  auc_roc  0.665884  0.031580
3   RNN              20    Temporal  auc_roc  0.650131  0.046905
4   RNN              20  Individual  auc_roc  0.702659  0.030884
Total de linhas extraídas: 60

✅ Arquivo salvo com sucesso: ood_aucroc_results.csv
